In [ ]:
import shutil
import os

drive_path = "/content/drive/MyDrive/FYP nawran"
vm_path = "/content/Dataset"

if os.path.exists(vm_path):
    print(f"Directory {vm_path} already exists. Skipping copy.")
else:
    print(f"Copying data from {drive_path} to {vm_path}...")
    shutil.copytree(drive_path, vm_path)
    print("Copy complete.")

Copying data from /content/drive/MyDrive/FYP nawran to /content/Dataset...
Copy complete.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install ultralytics opencv-python pandas tqdm

from ultralytics import YOLO
import pandas as pd
import cv2
import os
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from pathlib import Path
from pathlib import Path
import shutil
import numpy as np
import glob


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 72.3 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


Merge metadata and bounding boxes

In [ ]:


base = "/content/Dataset"

# Load the main metadata file
meta_df = pd.read_csv(f"{base}/CNRPark+EXT.csv")

# Load all camera bounding boxes
bbox_files = sorted(glob.glob(f"{base}/camera*.csv"))

merged_list = []

for cam_id, bbox_path in enumerate(bbox_files, start=1):
    bbox_df = pd.read_csv(bbox_path)

    # Convert camera number (1-9) to match the format used in metadata (A,B,C or numeric)
    camera_name = f"camera{cam_id}"

    # Filter metadata for this camera
    subset = meta_df[meta_df["image_url"].str.contains(camera_name, case=False)]

    # Merge on slot id
    combined = subset.merge(bbox_df, left_on="slot_id", right_on="SlotId", how="inner")
    combined["camera_name"] = camera_name
    merged_list.append(combined)

# Combine all cameras
merged_df = pd.concat(merged_list, ignore_index=True)
print("✅ Merged total rows:", len(merged_df))
merged_df.head()


/tmp/ipython-input-1426171502.py:4: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  meta_df = pd.read_csv(f"{base}/CNRPark+EXT.csv")


✅ Merged total rows: 144965


,camera,datetime,day,hour,image_url,minute,month,occupancy,slot_id,weather,year,occupant_changed,SlotId,X,Y,W,H,camera_name
0,01,2015-11-12_07.09,12,7,CNR-EXT/PATCHES/SUNNY/2015-11-12/camera1/S_201...,9,11,0,184,S,2015,NaN,184,2032,1652,240,240,camera1
1,01,2015-11-12_07.39,12,7,CNR-EXT/PATCHES/SUNNY/2015-11-12/camera1/S_201...,39,11,0,184,S,2015,NaN,184,2032,1652,240,240,camera1
2,01,2015-11-12_08.09,12,8,CNR-EXT/PATCHES/SUNNY/2015-11-12/camera1/S_201...,9,11,0,184,S,2015,NaN,184,2032,1652,240,240,camera1
3,01,2015-11-12_08.39,12,8,CNR-EXT/PATCHES/SUNNY/2015-11-12/camera1/S_201...,39,11,0,184,S,2015,NaN,184,2032,1652,240,240,camera1
4,01,2015-11-12_09.09,12,9,CNR-EXT/PATCHES/SUNNY/2015-11-12/camera1/S_201...,9,11,0,184,S,2015,NaN,184,2032,1652,240,240,camera1


In [ ]:


base = "/content/Dataset/FULL_IMAGE_1000x750"
count = 0
for root, dirs, files in os.walk(base):
    for f in files:
        if f.endswith(".jpg"):
            count += 1
            if count <= 5:
                print("Example:", os.path.join(root, f))
print(f"✅ Total .jpg images found: {count}")


Example: /content/Dataset/FULL_IMAGE_1000x750/RAINY/2015-12-22/camera5/2015-12-22_1323.jpg
Example: /content/Dataset/FULL_IMAGE_1000x750/RAINY/2015-12-22/camera5/2015-12-22_1623.jpg
Example: /content/Dataset/FULL_IMAGE_1000x750/RAINY/2015-12-22/camera5/2015-12-22_1153.jpg
Example: /content/Dataset/FULL_IMAGE_1000x750/RAINY/2015-12-22/camera5/2015-12-22_1553.jpg
Example: /content/Dataset/FULL_IMAGE_1000x750/RAINY/2015-12-22/camera5/2015-12-22_1223.jpg
✅ Total .jpg images found: 4278


SETUP YOLO DIRECTORY

In [ ]:
# Step 6 (FIXED)
base = "/content/Dataset/FULL_IMAGE_1000x750"
yolo_base = os.path.join(base, "YOLO_dataset")
images_output = os.path.join(yolo_base, "images")
labels_output = os.path.join(yolo_base, "labels")

os.makedirs(images_output, exist_ok=True)
os.makedirs(labels_output, exist_ok=True)


In [ ]:
# Step 4.1 – Collect all datetime keys from full images

import glob
from pathlib import Path

full_dt_keys = set()
full_img_paths = glob.glob(f"{base}/**/*.jpg", recursive=True)

for p in full_img_paths:
    fname = Path(p).name        # e.g. "2015-12-22_1323.jpg"
    stem = fname.replace(".jpg", "")
    full_dt_keys.add(stem)      # "2015-12-22_1323"

print("✅ Unique datetime keys in FULL_IMAGE_1000x750:", len(full_dt_keys))
print("Examples:", list(full_dt_keys)[:20])


✅ Unique datetime keys in FULL_IMAGE_1000x750: 3334
Examples: ['2016-01-09_0755', '2015-11-21_0902', '2016-02-12_1542', '2016-01-15_1635', '2016-01-15_0837', '2015-11-22_0736', '2015-12-03_1320', '2015-12-18_1726', '2016-01-14_1132', '2016-01-08_1010', '2015-11-12_1116', '2015-12-18_0726', '2016-01-14_0904', '2015-11-25_1646', '2015-12-18_1421', '2015-12-17_0851', '2015-12-22_0824', '2015-12-10_1218', '2015-11-21_1117', '2015-11-25_0940']


In [ ]:
# Step 4.2 – Normalize datetime in merged_df to dt_key and filter to only existing full images

def make_dt_key(dt_str, img_url):
    dt_str = str(dt_str)

    # EXT format: "2015-11-12_07.09"
    if "CNR-EXT" in str(img_url) and "-" in dt_str and "." in dt_str:
        date_part, time_part = dt_str.split("_")   # "2015-11-12", "07.09"
        time_part = time_part.replace(".", "")     # "0709"
        return f"{date_part}_{time_part}"          # "2015-11-12_0709"

    # Park format: "20150703_0805" (no '-'s)
    if "CNRPark" in str(img_url) and "-" not in dt_str:
        date_part, time_part = dt_str.split("_")   # "20150703", "0805"
        y = date_part[0:4]
        m = date_part[4:6]
        d = date_part[6:8]
        return f"{y}-{m}-{d}_{time_part}"          # "2015-07-03_0805"

    # fallback: return as-is
    return dt_str

merged_df["dt_key"] = merged_df.apply(
    lambda r: make_dt_key(r["datetime"], r["image_url"]),
    axis=1
)

print("Before filtering:", len(merged_df))
merged_df = merged_df[ merged_df["dt_key"].isin(full_dt_keys) ].reset_index(drop=True)
print("After filtering to dt_keys that have full images:", len(merged_df))

merged_df[["camera", "camera_name", "datetime", "dt_key", "image_url"]].head()


Before filtering: 144965
After filtering to dt_keys that have full images: 144965


,camera,camera_name,datetime,dt_key,image_url
0,01,camera1,2015-11-12_07.09,2015-11-12_0709,CNR-EXT/PATCHES/SUNNY/2015-11-12/camera1/S_201...
1,01,camera1,2015-11-12_07.39,2015-11-12_0739,CNR-EXT/PATCHES/SUNNY/2015-11-12/camera1/S_201...
2,01,camera1,2015-11-12_08.09,2015-11-12_0809,CNR-EXT/PATCHES/SUNNY/2015-11-12/camera1/S_201...
3,01,camera1,2015-11-12_08.39,2015-11-12_0839,CNR-EXT/PATCHES/SUNNY/2015-11-12/camera1/S_201...
4,01,camera1,2015-11-12_09.09,2015-11-12_0909,CNR-EXT/PATCHES/SUNNY/2015-11-12/camera1/S_201...


In [ ]:
import glob

base_path = "/content/Dataset/FULL_IMAGE_1000x750"
imgs = glob.glob(f"{base_path}/**/*.jpg", recursive=True)

print(f"✅ Found {len(imgs)} images in total")
print("Example paths:")
for p in imgs[:5]:
    print(p)


✅ Found 4278 images in total
Example paths:
/content/Dataset/FULL_IMAGE_1000x750/RAINY/2015-12-22/camera5/2015-12-22_1323.jpg
/content/Dataset/FULL_IMAGE_1000x750/RAINY/2015-12-22/camera5/2015-12-22_1623.jpg
/content/Dataset/FULL_IMAGE_1000x750/RAINY/2015-12-22/camera5/2015-12-22_1153.jpg
/content/Dataset/FULL_IMAGE_1000x750/RAINY/2015-12-22/camera5/2015-12-22_1553.jpg
/content/Dataset/FULL_IMAGE_1000x750/RAINY/2015-12-22/camera5/2015-12-22_1223.jpg


In [ ]:
print("images_dir =", base)
print("len(merged_df) =", len(merged_df))
print(merged_df[['image_url']].head())


images_dir = /content/Dataset/FULL_IMAGE_1000x750
len(merged_df) = 144965
                                           image_url
0  CNR-EXT/PATCHES/SUNNY/2015-11-12/camera1/S_201...
1  CNR-EXT/PATCHES/SUNNY/2015-11-12/camera1/S_201...
2  CNR-EXT/PATCHES/SUNNY/2015-11-12/camera1/S_201...
3  CNR-EXT/PATCHES/SUNNY/2015-11-12/camera1/S_201...
4  CNR-EXT/PATCHES/SUNNY/2015-11-12/camera1/S_201...


In [ ]:
import glob

base_path = "/content/Dataset/FULL_IMAGE_1000x750"
samples = glob.glob(f"{base_path}/**/*.jpg", recursive=True)

print("Total images found:", len(samples))
samples[:20]  # show first 20 filenames


Total images found: 4278


['/content/Dataset/FULL_IMAGE_1000x750/RAINY/2015-12-22/camera5/2015-12-22_1323.jpg',
 '/content/Dataset/FULL_IMAGE_1000x750/RAINY/2015-12-22/camera5/2015-12-22_1623.jpg',
 '/content/Dataset/FULL_IMAGE_1000x750/RAINY/2015-12-22/camera5/2015-12-22_1153.jpg',
 '/content/Dataset/FULL_IMAGE_1000x750/RAINY/2015-12-22/camera5/2015-12-22_1553.jpg',
 '/content/Dataset/FULL_IMAGE_1000x750/RAINY/2015-12-22/camera5/2015-12-22_1223.jpg',
 '/content/Dataset/FULL_IMAGE_1000x750/RAINY/2015-12-22/camera5/2015-12-22_0853.jpg',
 '/content/Dataset/FULL_IMAGE_1000x750/RAINY/2015-12-22/camera5/2015-12-22_0723.jpg',
 '/content/Dataset/FULL_IMAGE_1000x750/RAINY/2015-12-22/camera5/2015-12-22_1023.jpg',
 '/content/Dataset/FULL_IMAGE_1000x750/RAINY/2015-12-22/camera5/2015-12-22_1053.jpg',
 '/content/Dataset/FULL_IMAGE_1000x750/RAINY/2015-12-22/camera5/2015-12-22_1523.jpg',
 '/content/Dataset/FULL_IMAGE_1000x750/RAINY/2015-12-22/camera5/2015-12-22_1253.jpg',
 '/content/Dataset/FULL_IMAGE_1000x750/RAINY/2015-12-2

CONVERT TO YOLOv8 ANNOTATION FORMAT

In [ ]:
import glob
import cv2
from pathlib import Path
from tqdm import tqdm

# Original full resolution from CNRPark+EXT docs
ORIG_W, ORIG_H = 2592, 1944

processed = 0
valid_images = set()

print("images_dir =", base)
print("Example dt_keys:", merged_df["dt_key"].head().tolist())

for _, row in tqdm(merged_df.iterrows(), total=len(merged_df)):

    dt_key = row["dt_key"]  # e.g. "2015-11-12_0709"

    # 1) Find corresponding full image
    matches = glob.glob(f"{base}/**/{dt_key}.jpg", recursive=True)
    if not matches:
        continue
    img_path = matches[0]

    img = cv2.imread(img_path)
    if img is None:
        continue

    h, w = img.shape[:2]    # should be 750 x 1000

    # Sanity (optional, run a few times then you can comment this)
    # print("Loaded size:", w, h)  # expect (1000, 750)

    # 2) SCALE bbox from 2592x1944 to 1000x750
    scale_x = w / ORIG_W
    scale_y = h / ORIG_H

    X_orig = row["X"]
    Y_orig = row["Y"]
    W_orig = row["W"]
    H_orig = row["H"]

    X = X_orig * scale_x
    Y = Y_orig * scale_y
    W_box = W_orig * scale_x
    H_box = H_orig * scale_y

    # 3) Compute normalized YOLO coords
    x_center = (X + W_box / 2) / w
    y_center = (Y + H_box / 2) / h
    bw = W_box / w
    bh = H_box / h

    # 4) Clip to [0,1] just in case of rounding at borders
    x_center = min(max(x_center, 0.0), 1.0)
    y_center = min(max(y_center, 0.0), 1.0)
    bw = min(max(bw, 0.0), 1.0)
    bh = min(max(bh, 0.0), 1.0)

    cls = int(row["occupancy"])   # 0 = free, 1 = occupied

    # 5) Label filename matches image filename
    base_name = Path(img_path).stem          # e.g. "2015-11-12_0709"
    label_file = os.path.join(labels_output, base_name + ".txt")

    with open(label_file, "a") as f:
        f.write(f"{cls} {x_center} {y_center} {bw} {bh}\n")

    # 6) Copy image once
    dest_img = os.path.join(images_output, base_name + ".jpg")
    if not os.path.exists(dest_img):
        cv2.imwrite(dest_img, img)

    valid_images.add(dest_img)
    processed += 1

print(f"✅ Slot annotations generated: {processed}")
print(f"✅ Unique full-frame training images: {len(valid_images)}")


images_dir = /content/Dataset/FULL_IMAGE_1000x750
Example dt_keys: ['2015-11-12_0709', '2015-11-12_0739', '2015-11-12_0809', '2015-11-12_0839', '2015-11-12_0909']


100%|██████████| 144965/144965 [36:07<00:00, 66.88it/s]

✅ Slot annotations generated: 144965
✅ Unique full-frame training images: 3174


SPLIT DATA INTO TRAIN/VAL

In [ ]:
from sklearn.model_selection import train_test_split

valid_images = list(valid_images)
print("Total valid images with labels:", len(valid_images))

if len(valid_images) == 0:
    raise RuntimeError("❌ No valid images were generated. Check Step 8.")

for split_dir in ["train", "val"]:
    os.makedirs(os.path.join(yolo_base, "images", split_dir), exist_ok=True)
    os.makedirs(os.path.join(yolo_base, "labels", split_dir), exist_ok=True)

train_imgs, val_imgs = train_test_split(valid_images, test_size=0.2, random_state=42)

for split, imgs in zip(["train", "val"], [train_imgs, val_imgs]):
    split_img_dir = os.path.join(yolo_base, "images", split)
    split_lbl_dir = os.path.join(yolo_base, "labels", split)

    for img_path in imgs:
        img_name = os.path.basename(img_path)
        base_name = os.path.splitext(img_name)[0]
        src_label = os.path.join(labels_output, base_name + ".txt")

        if not os.path.exists(src_label):
            continue

        shutil.copy(img_path, os.path.join(split_img_dir, img_name))
        shutil.copy(src_label, os.path.join(split_lbl_dir, base_name + ".txt"))

print("✅ Finished splitting into train/val.")
print("   Train images:", len(train_imgs))
print("   Val images:", len(val_imgs))


Total valid images with labels: 3174
✅ Finished splitting into train/val.
   Train images: 2539
   Val images: 635


In [ ]:
data_yaml_path = os.path.join(yolo_base, "data.yaml")

with open(data_yaml_path, "w") as f:
    f.write(f"path: {yolo_base}\n")
    f.write("train: images/train\n")
    f.write("val: images/val\n")
    f.write("names:\n")
    f.write("  0: free\n")
    f.write("  1: occupied\n")

print("✅ data.yaml written at:", data_yaml_path)


✅ data.yaml written at: /content/Dataset/FULL_IMAGE_1000x750/YOLO_dataset/data.yaml


Training

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8s.pt")  # or yolov8s.pt

model.train(
    data=data_yaml_path,
    epochs=50,
    imgsz=640,
    batch=16,
    name="parking_yolov8"
)


Ultralytics 8.3.228 🚀 Python-3.12.12 torch-2.8.0+cu126 CUDA:0 (NVIDIA A100-SXM4-40GB, 40507MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/Dataset/FULL_IMAGE_1000x750/YOLO_dataset/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=parking_yolov8, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=Tr

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7902e17acb90>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.04804

In [ ]:
!ls /content/runs/detect/parking_yolov8/weights


best.pt  last.pt


In [ ]:
from google.colab import files
files.download('/content/runs/detect/parking_yolov8/weights/best.pt')


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>